In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
qdrant_api_key = os.getenv("qdrant_api_key")
gemini_api_key = os.getenv("gemini_api_key")
mistal_api_key = os.getenv("mistal_api_key")

# Load model embedding

In [2]:

import torch
import tqdm


from sentence_transformers import SentenceTransformer
from tqdm import tqdm

class BGEEmbedder:
    def __init__(self, model_name="BAAI/bge-m3"):
        self.model = SentenceTransformer(model_name)
        self.prefix = "Represent this sentence for searching relevant passages: "

    def embed(self, texts, batch_size=16):
        # BGE-M3 yêu cầu prefix cho truy vấn và văn bảna
        texts_with_prefix = [
            text if text.startswith(self.prefix) else self.prefix + text
            for text in texts
        ]

        # Dùng encode với batch size và normalize sẵn
        embeddings = self.model.encode(
            texts_with_prefix,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=True
        )
        return embeddings




In [3]:
# from sentence_transformers import SentenceTransformer

# # Load model BGE-M3
# model = SentenceTransformer("BAAI/bge-m3")

# def embed_fn(text: str):
#     # BGE-M3 khuyến nghị prefix truy vấn bằng "Represent this sentence for searching relevant passages:"
#     if not text.startswith("Represent"):
#         text = "Represent this sentence for searching relevant passages: " + text
#     return model.encode(text, normalize_embeddings=True)


# Gọi Qdrant vector DB

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from qdrant_client.models import Filter
client = QdrantClient(
    url=r"https://0f47d391-b7c1-45d9-a956-5f7228cd80f3.europe-west3-0.gcp.cloud.qdrant.io:6333",
    api_key=qdrant_api_key,
    prefer_grpc=False
)
# client = QdrantClient(
#     host="localhost",
#     port=6333
# )

# client.recreate_collection(
#     collection_name="flm_fap",
#     vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
# )
# client.delete(
#     collection_name="flm_fap",
#     points_selector=Filter(must=[])  # Xoá toàn bộ points
# )
client.get_collection("FINAL").payload_schema


{'subject_code': PayloadIndexInfo(data_type=<PayloadSchemaType.KEYWORD: 'keyword'>, params=None, points=6393),
 'semester': PayloadIndexInfo(data_type=<PayloadSchemaType.INTEGER: 'integer'>, params=None, points=65),
 'type': PayloadIndexInfo(data_type=<PayloadSchemaType.KEYWORD: 'keyword'>, params=None, points=6611)}

In [5]:
embedder = BGEEmbedder()  

# Demo search

## Tạo subject map cho các môn học

In [6]:
import pandas as pd
df_flm=pd.read_csv(r'D:\Learn\Semester_5\SEG301\Fap-Chat\data\DATA cố định\FLM\FINAL\FINAL_DF_FLM.csv')
subject_map = {
    row["SubjectCode"]: f"{row["SubjectCode"]} - {row["Subject Name"]}"
    for _, row in df_flm[["SubjectCode", "Subject Name"]].dropna().drop_duplicates().iterrows()
}
subject_embeddings = {
    code: embedder.embed([name])[0]
    for code, name in subject_map.items()
}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
subject_map

{'OTP101': 'OTP101 - Orientation and General Training Program_Định hướng và Rèn luyện tập trung',
 'PEN': 'PEN - Preparation English_Tiếng Anh chuẩn bị',
 'PHE_COM*1': 'PHE_COM*1 - Physical Education 1_Giáo dục thể chất 1',
 'VOV114': 'VOV114 - Vovinam 1',
 'COV111': 'COV111 - Cờ Vua 1_Chess 1',
 'TMI_ELE': 'TMI_ELE - Traditional musical instrument_Nhạc cụ truyền thống',
 'DTR103': 'DTR103 - Nhạc cụ truyền thống-Đàn Tranh',
 'DBA103': 'DBA103 - Nhạc cụ truyền thống - Đàn Bầu',
 'DSA103': 'DSA103 - Nhạc cụ truyền thống- Sáo trúc',
 'DNG103': 'DNG103 - Nhạc cụ truyền thống-Đàn nguyệt',
 'DTB103': 'DTB103 - Nhạc cụ truyền thống- Đàn Tỳ bà',
 'TRG103': 'TRG103 - Nhạc cụ truyền thống -Trống dân tộc',
 'DNH103': 'DNH103 - Nhạc cụ truyền thống- Đàn Nhị',
 'CSI106': 'CSI106 - Introduction to Computer Science_Nhập môn khoa học máy tính',
 'MAD101': 'MAD101 - Discrete mathematics_Toán rời rạc',
 'MAE101': 'MAE101 - Mathematics for Engineering_Toán cho ngành kỹ thuật',
 'PFP191': 'PFP191 - Progr

In [8]:
# Xoá trong subject_map
subject_map = {
    k: v for k, v in subject_map.items()
    if not k.startswith("PHE_COM")
}

# Xoá trong embeddings
subject_embeddings = {
    k: v for k, v in subject_embeddings.items()
    if not k.startswith("PHE_COM")
}
# Xoá trong subject_map
subject_map = {
    k: v for k, v in subject_map.items()
    if not k.startswith("AI17_COM")
}

# Xoá trong embeddings
subject_embeddings = {
    k: v for k, v in subject_embeddings.items()
    if not k.startswith("AI17_COM")
}
# Xoá trong subject_map
subject_map = {
    k: v for k, v in subject_map.items()
    if not k.startswith("AI17_GRA_ELE")
}

# Xoá trong embeddings
subject_embeddings = {
    k: v for k, v in subject_embeddings.items()
    if not k.startswith("AI17_GRA_ELE")
}


In [9]:
subject_map

{'OTP101': 'OTP101 - Orientation and General Training Program_Định hướng và Rèn luyện tập trung',
 'PEN': 'PEN - Preparation English_Tiếng Anh chuẩn bị',
 'VOV114': 'VOV114 - Vovinam 1',
 'COV111': 'COV111 - Cờ Vua 1_Chess 1',
 'TMI_ELE': 'TMI_ELE - Traditional musical instrument_Nhạc cụ truyền thống',
 'DTR103': 'DTR103 - Nhạc cụ truyền thống-Đàn Tranh',
 'DBA103': 'DBA103 - Nhạc cụ truyền thống - Đàn Bầu',
 'DSA103': 'DSA103 - Nhạc cụ truyền thống- Sáo trúc',
 'DNG103': 'DNG103 - Nhạc cụ truyền thống-Đàn nguyệt',
 'DTB103': 'DTB103 - Nhạc cụ truyền thống- Đàn Tỳ bà',
 'TRG103': 'TRG103 - Nhạc cụ truyền thống -Trống dân tộc',
 'DNH103': 'DNH103 - Nhạc cụ truyền thống- Đàn Nhị',
 'CSI106': 'CSI106 - Introduction to Computer Science_Nhập môn khoa học máy tính',
 'MAD101': 'MAD101 - Discrete mathematics_Toán rời rạc',
 'MAE101': 'MAE101 - Mathematics for Engineering_Toán cho ngành kỹ thuật',
 'PFP191': 'PFP191 - Programming Fundamentals with Python_Cơ sở lập trình với Python',
 'VOV124'

## Đây là hàm để chọn ra top các môn học liên quan dựa vào việc so sánh querry với các subject map mới tạo ở trên

ae thấy có cái top_k là top các môn học nên trả ra

In [10]:
from numpy import dot
from numpy.linalg import norm
import heapq


def detect_subject(query, top_k=2, threshold=0.7):
    query_vec = embedder.embed([query])[0]
    
    sims = {
        code: dot(query_vec, emb) / (norm(query_vec) * norm(emb))
        for code, emb in subject_embeddings.items()
    }

    top_subjects = heapq.nlargest(top_k, sims.items(), key=lambda x: x[1])

    # Chỉ giữ lại những cái vượt ngưỡng
    top_subjects = [(code, score) for code, score in top_subjects if score >= threshold]

    # Nếu không có gì đạt ngưỡng → trả về danh sách rỗng
    return top_subjects



## Hàm dịch

In [11]:
from deep_translator import GoogleTranslator

def translate_vi_to_en_google(text):
    return GoogleTranslator(source='vi', target='en').translate(text)

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# Load mô hình dịch
model_id = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Pipeline dịch offline
translator = pipeline(
    "translation",
    model=model,
    tokenizer=tokenizer,
    src_lang="vie",
    tgt_lang="eng_Latn",
    max_length=512
)

def translate_vi_to_en_local(text):
    result = translator(text)
    return result[0]['translation_text']



Device set to use cpu


## Type_map đây ae

In [29]:
TYPE_DESCRIPTIONS = {
    "overview": "queries about which subjects match certain characteristics (e.g., taught in a specific semester, related to a topic, or general overviews of subject goals, credits, syllabus, or curriculum structure.",
    "construtive_question": "thought-provoking questions that encourage critical thinking or reflection.",
    "assessment": "evaluations, types of tests, exams, final exams (FE), progress exams (PE), term exams (TE), and grading weights.",
    "session": "lecture sessions, lessons, and topics covered in each week or session.",
    "material": "recommended textbooks, reference materials, lecture slides, readings, or other learning resources.",
    "learning outcome": "expected knowledge, skills, or competencies students should gain after completing the course.",
    "guide": "instructions or guidance for students on how to complete tasks, assignments, projects, or how to use certain tools or platforms.",
    "student_list": "list of students enrolled in the course, including names, student IDs, and email addresses.",
    # "attendance": "records of student attendance per session, including presence or absence, date, room, and instructor.",
    # "grade detail": "detailed grade components, including evaluation item name, category, weight, and obtained score.",
    # "course summary": "final course grade summary including average score, status, and summary notes about course performance.",
    # "student profile": "personal profile of an unique student, including full name, student ID, email, program, and major."
}

TYPE_KEYWORDS = {
    "overview": ["overview", "objective", "goal", "credits", "semester", "prerequisite", "syllabus", "subject", "subjects", 'general'],
    "construtive_question": ["why", "what if", "critical", "discussion", "reflect", "ethical", "opinion", "thinking"],
    "assessment": ["exam", "test", "quiz", "grading", "project", "evaluation", "score", "mark", "weight", "assessment"],
    "session": ["week", "lesson", "lecture", "topic", "schedule", "session", "class", "timetable"],
    "material": ["textbook", "slide", "document", "reading", "reference", "material", "resource", "book", "pdf", "file"],
    "learning outcome": ["learn", "outcome", "skill", "competency", "ability", "achieve", "knowledge", "CLO", "LO", "learning outcome"],
    "student_list": ["student", "id", "mssv", "email", "class list", "enrolled", "danh sách sinh viên", "học sinh", "danh sách"],
    "guide": ["how to", "instruction", "guide", "tutorial", "step", "steps", "do", "complete", "submit", "platform", "tool", "usage", "help", "assist", "support", "direction"],
    # "attendance": ["attendance", "present", "absent", "record", "check-in", "participation", "presence", "ca học", "phòng", "giảng viên"],
    # "grade detail": ["score", "mark", "value", "grade", "item", "category", "evaluation", "component", "trọng số", "điểm"],
    # "course summary": ["summarize","summary", "average", "final", "result", "status", "performance", "overall", "total", "điểm trung bình", "kết quả"],
    # "student profile": ["student", "profile", "id", "name", "email", "program", "major", "class", "course", "personal"]
}



type_embeddings = {
    t: embedder.embed([desc])[0] for t, desc in TYPE_DESCRIPTIONS.items()
}

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## Hàm chạy chính (ko có api GEMINI)

In [13]:
from numpy import dot
from numpy.linalg import norm
from collections import defaultdict
import re
# from rank_bm25 import BM25Okapi

# # Tokenize mô tả
# type_docs = [desc.lower().split() for desc in TYPE_DESCRIPTIONS.values()]
# bm25 = BM25Okapi(type_docs)
# type_list = list(TYPE_DESCRIPTIONS.keys())



# # --- Step 0: Optional Translate Query (nếu cần)
# query = "tổng quan môn search engine"

# # Nếu có hàm dịch: query_en = translate_vi_to_en_nllb(query)
# query_en = query  # nếu bạn không cần dịch hoặc đã là tiếng Anh

# --- Step 1: Detect type bằng embedding
# def detect_type_by_embedding(query_en):
#     query_vec = embedder.embed([query_en])[0]
#     sims = {
#         t: dot(query_vec, vec) / (norm(query_vec) * norm(vec))
#         for t, vec in type_embeddings.items()
#     }
#     print(sims)
#     best_type = max(sims, key=sims.get)

#     return best_type 
def detect_type_by_embedding(query_en, alpha=0.8, beta=0.2):
    query_vec = embedder.embed([query_en])[0]
    
    # 1. Embedding similarity
    sims = {
        t: dot(query_vec, vec) / (norm(query_vec) * norm(vec))
        for t, vec in type_embeddings.items()
    }

    # 2. Keyword score
    keyword_scores = defaultdict(int)
    query_lower = query_en.lower()
    for t, keywords in TYPE_KEYWORDS.items():
        for kw in keywords:
            if re.search(rf"\b{re.escape(kw)}\b", query_lower):
                keyword_scores[t] += 1

    # 3. Normalize keyword scores
    max_kw = max(keyword_scores.values(), default=1)
    keyword_scores_norm = {
        t: keyword_scores[t] / max_kw if max_kw > 0 else 0
        for t in type_embeddings
    }

    # 4. Combine scores
    final_scores = {
        t: alpha * sims.get(t, 0) + beta * keyword_scores_norm.get(t, 0)
        for t in type_embeddings
    }

    print("Similarity scores:", sims)
    print("Keyword scores:", dict(keyword_scores))
    print("Final combined scores:", final_scores)

    best_type = max(final_scores, key=final_scores.get)
    return best_type
# def detect_type_by_embedding(query_en, alpha=0.8, beta=0.2):
#     # 1. Embedding similarity
#     query_vec = embedder.embed([query_en])[0]
#     sims = {
#         t: dot(query_vec, vec) / (norm(query_vec) * norm(vec))
#         for t, vec in type_embeddings.items()
#     }

#     # 2. BM25 keyword-based score
#     query_tokens = query_en.lower().split()
#     bm25_scores = bm25.get_scores(query_tokens)
#     max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
#     bm25_scores_norm = {type_list[i]: bm25_scores[i] / max_bm25 for i in range(len(type_list))}

#     # 3. Kết hợp
#     final_scores = {
#         t: alpha * sims.get(t, 0) + beta * bm25_scores_norm.get(t, 0)
#         for t in type_list
#     }

#     print("Cosine scores:", sims)
#     print("BM25 scores:", bm25_scores_norm)
#     print("Combined scores:", final_scores)

#     best_type = max(final_scores, key=final_scores.get)
#     return best_type

# query_type = detect_type_by_embedding(query_en)


# query = "môn học được bonus point nhờ bài báo được accepted"
# query_en = query  # hoặc dùng translate nếu cần
# query_en = translate_vi_to_en_google(query)
def search_embedding(query=''):
    
    query_en = translate_vi_to_en_google(query)

    print('I/--------------Querry sau khi dịch:---------------')
    print(query_en)

    print('II/--------------Xác định chủ đề của querry---------------')
    detected_type = detect_type_by_embedding(query_en)
    print('III/--------------Xác định môn học của querry (nếu có)---------------')
    detected_subject = detect_subject(query_en)
    
    list_not_subject=['student_list','guide','student profile', 'attendance','course summary']
    if detected_type in list_not_subject:
        detected_subject=[]
        query_vec = embedder.embed([query])[0]
    else: 
        query_vec = embedder.embed([query_en])[0]

    # query_vec = embedder.embed([query_en])[0]


    # query_filter = {"must": []}
    # if detected_type:
    #     query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    # if detected_subject:
    #     query_filter["must"].append({"key": "subject_code", "match": {"value": detected_subject}})

    query_filter = {"should": [], "must":[]}
    if detected_type:
        print("Loại detect được là:")
        print(detected_type)
        query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    if detected_subject:
        for subject_code in detected_subject:
            print(subject_code[0])
            query_filter["should"].append({
                "key": "subject_code",
                "match": {"value": subject_code[0]}
            })
    
    hits = client.search(
        collection_name="FINAL",
        query_vector=query_vec.tolist(),
        limit=10,
        query_filter=query_filter if query_filter["must"] or query_filter["should"] else None
    )
    # print("""⭐ Top 1
    # 🔍 Score: 0.7008
    # 📘 Subject: None - student_list
    # 📄 Content:
    # Họ và tên: Đỗ Tuấn Minh. Mã số sinh viên: DE190415. Email: tuanminhdo1203@gmail.com.""")

    # --- Step 4: Hiển thị kết quả
    for i, hit in enumerate(hits, 1):  # bắt đầu từ 1
        print(f"\n⭐ Top {i}")
        print(f"🔍 Score: {hit.score:.4f}")
        print(f"📘 Subject: {hit.payload.get('subject_code')} - {hit.payload.get('type')}")
        print(f"📄 Content:\n{hit.payload.get('content')}")
        print("--------")
    print(hits)

In [14]:
search_embedding("điểm thi cuối kì CPV")

I/--------------Querry sau khi dịch:---------------
The final test score of CPV
II/--------------Xác định chủ đề của querry---------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Similarity scores: {'overview': 0.5238662, 'construtive_question': 0.5237905, 'assessment': 0.6375254, 'session': 0.53356767, 'material': 0.57411665, 'learning outcome': 0.5739994, 'guide': 0.53900725, 'student_list': 0.53780717, 'attendance': 0.5063547, 'grade detail': 0.6197459, 'course summary': 0.6744237, 'student profile': 0.53412855}
Keyword scores: {'assessment': 2, 'grade detail': 1, 'course summary': 1, 'overview': 0, 'construtive_question': 0, 'session': 0, 'material': 0, 'learning outcome': 0, 'guide': 0, 'student_list': 0, 'attendance': 0, 'student profile': 0}
Final combined scores: {'overview': 0.41909294128417973, 'construtive_question': 0.41903238296508794, 'assessment': 0.7100203037261963, 'session': 0.42685413360595703, 'material': 0.4592933177947998, 'learning outcome': 0.4591995239257813, 'guide': 0.4312057971954346, 'student_list': 0.43024573326110843, 'attendance': 0.40508375167846683, 'grade detail': 0.5957967281341553, 'course summary': 0.6395389556884765, 'stud

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loại detect được là:
assessment


C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_29872\851659152.py:141: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(



⭐ Top 1
🔍 Score: 0.7000
📘 Subject: CPV301 - assessment
📄 Content:
TYPE: assessment
Subject: CPV301 - Computer Vision_Thị giác máy tính
Category: Final exam | Part: 1 | Weight: 30.0%
Question Type: Multiple choices Marked by Computer
Knowledge and Skill: All subjects in syllabus
Grading Guide: By Exam Board
Completion Criteria: 4
Duration: 60'/each
Note: The exam questions must be updated or different at least 70% to the previous ones.
--------

⭐ Top 2
🔍 Score: 0.6830
📘 Subject: CPV301 - assessment
📄 Content:
TYPE: assessment
Subject: CPV301 - Computer Vision_Thị giác máy tính
Category: Practical Exam | Part: 1 | Weight: 20.0%
Question Type: N/A
Knowledge and Skill: N/A
Grading Guide: in class, by teacher
Completion Criteria: >0
Duration: 90'/each
Note: N/A
--------

⭐ Top 3
🔍 Score: 0.6785
📘 Subject: VNR202 - assessment
📄 Content:
TYPE: assessment
Subject: VNR202 - History of CPV_Lịch sử Đảng Cộng sản Việt Nam
Category: Final exam | Part: 1 | Weight: 30.0%
Question Type: Câu hỏi trắc

### Đánh giá

In [37]:
import json
from tqdm import tqdm

COLLECTION_NAME = "FINAL"

with open(r"D:\Learn\Semester_5\SEG301\DEMO_local\data_eval.json", "r", encoding="utf-8") as f:
    query_data = json.load(f)

len(query_data)

# result_logs = []

# # Hàm lấy nội dung chunk từ Qdrant
# def get_chunk_content(point_id):
#     response = client.retrieve(
#         collection_name=COLLECTION_NAME,
#         ids=[point_id],
#         with_payload=True
#     )
#     if response:
#         return response[0].payload.get("content", "")
#     return "[Không tìm thấy nội dung]"

# def search_for_evaluation_with_manual_filter(query, filter_dict):
#     query_vec = embedder.embed([query])[0]

#     query_filter = {"must": [], "should": []}
#     for key, value in filter_dict.items():
#         query_filter["must"].append({
#             "key": key,
#             "match": {"value": value}
#         })

#     hits = client.search(
#         collection_name=COLLECTION_NAME,
#         query_vector=query_vec.tolist(),
#         limit=10,
#         query_filter=query_filter if query_filter["must"] else None,
#         with_payload=True
#     )

#     return [hit.id for hit in hits]


# # Gán nhãn
# for item in tqdm(query_data):
#     query = item["query_text"]

#     # 👉 In câu hỏi trước
#     print(f"\n📌 Câu hỏi: {query}")

#     # Nhập filter sau khi đã xem câu hỏi
#     filter_dict = {}
#     type_inp = input("👉 Nhập type (hoặc Enter để bỏ qua): ").strip()
#     if type_inp:
#         filter_dict["type"] = type_inp

#     subject_inp = input("👉 Nhập subject_code (hoặc Enter để bỏ qua): ").strip()
#     if subject_inp:
#         filter_dict["subject_code"] = subject_inp

#     semester_inp = input("👉 Nhập semester (hoặc Enter để bỏ qua): ").strip()
#     if semester_inp:
#         filter_dict["semester"] = semester_inp

#     # Tìm kiếm
#     ranked_ids = search_for_evaluation_with_manual_filter(query, filter_dict)

#     # Hiển thị kết quả
#     for i, cid in enumerate(ranked_ids, 1):
#         chunk_text = get_chunk_content(cid)
#         print(f"\n{i}. Chunk ID: {cid}")
#         print(f"Nội dung: {chunk_text[:300]}...")

#     # Gán nhãn
#     print(f"\n📌 Lặp lại câu hỏi: {query}")
#     selected = int(input("🎯 Chọn chunk đúng (1-10 hoặc 0 nếu không có): "))
#     correct_id = ranked_ids[selected - 1] if 1 <= selected <= len(ranked_ids) else None

#     result_logs.append({
#         "query_text": query,
#         "filters": filter_dict,
#         "ranked_ids": ranked_ids,
#         "correct_id": correct_id
#     })

# # Lưu lại

# with open("result_with_labels.json", "w", encoding="utf-8") as f:
#     json.dump(result_logs, f, ensure_ascii=False, indent=2)


33

## GEMINI tóm tắt

In [ ]:
# --- CÁC IMPORT CẦN THIẾT ---
from numpy import dot
from numpy.linalg import norm
from collections import defaultdict
import re
import requests
import json
import time # [THÊM MỚI] Import thư viện time

# --- HÀM TÓM TẮT (GIỮ NGUYÊN NHƯ CODE GỐC CỦA BẠN) ---
def summarize_with_gemini( api_key: str, model: str = "models/gemini-2.0-flash", retrieved_chunks='', user_query='') -> str:
    """
    Tóm tắt nội dung sử dụng Gemini API qua REST request.
    """
    url = f"https://generativelanguage.googleapis.com/v1beta/{model}:generateContent?key={api_key}"
    headers = {"Content-Type": "application/json"}
    
    prompt = f"""
    Bạn là một trợ lý AI có nhiệm vụ trả lời câu hỏi của người dùng dựa trên các đoạn thông tin đã được truy xuất từ tài liệu.

    Dưới đây là nội dung truy xuất:

    ==== context ====
    {retrieved_chunks}
    ===================

    Câu hỏi của người dùng:
    {user_query}

    Yêu cầu:
    - Trả lời sát với câu hỏi người dùng
    - Chỉ sử dụng thông tin có trong phần "context" để trả lời. Có thể điều chỉnh cách ghi lại cho đẹp
    - Nếu câu hỏi yêu cầu nhóm, liệt kê, hoặc so sánh thì hãy xử lý và tổng hợp từ các đoạn context.
    

    Trả lời bằng văn phong ngắn gọn, rõ ràng, chính xác. Trình bày rõ ràng đừng hiện các kí tự của markdown
    Có thể đưa ra thông tin gần đúng nếu bạn không chắc và bạn phải cảnh báo điều đó
    """

    
    data = {"contents": [{"parts": [{"text": prompt}]}]}
    
    try:
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        result = response.json()
        summary = result["candidates"][0]["content"]["parts"][0]["text"]
        return summary.strip()
    except requests.exceptions.RequestException as e:
        print(f"Lỗi gọi API: {e}")
        return "Lỗi khi gọi API tóm tắt."
    except (KeyError, IndexError) as e:
        print(f"Lỗi khi xử lý response từ API: {e}. Response: {response.text}")
        return "Không thể tóm tắt nội dung do lỗi xử lý response."
    
    
def search(querry=''):    
    # --- PHẦN CODE RAG CỦA BẠN (GIỮ NGUYÊN) ---
    # ... (Giả sử toàn bộ phần code xử lý query và tìm kiếm `hits` của bạn nằm ở đây) ...
    query = querry
    # query_en = query  # hoặc dùng translate nếu cần
    query_en = translate_vi_to_en_local(query)
    # query_en = translate_vi_to_en_google(query)
    print('I/--------------Querry sau khi dịch:---------------')
    print(query_en)

    print('II/--------------Xác định chủ đề của querry---------------')
    detected_type = detect_type_by_embedding(query_en)
    print('III/--------------Xác định môn học của querry (nếu có)---------------')
    detected_subject = detect_subject(query_en)
    if detected_type=='student_list':
        query_vec = embedder.embed([query])[0]
    else:
        query_vec = embedder.embed([query_en])[0]


    # query_filter = {"must": []}
    # if detected_type:
    #     query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    # if detected_subject:
    #     query_filter["must"].append({"key": "subject_code", "match": {"value": detected_subject}})

    query_filter = {"should": [], "must":[]}
    if detected_type:
        print("Loại detect được là:")
        print(detected_type)
        query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    if detected_subject:
        for subject_code in detected_subject:
            print(subject_code[0])
            query_filter["should"].append({
                "key": "subject_code",
                "match": {"value": subject_code[0]}
            })

    hits = client.search(
        collection_name="flm_fap",
        query_vector=query_vec.tolist(),
        limit=50,
        query_filter=query_filter if query_filter["must"] or query_filter["should"] else None
    )
    # print("""⭐ Top 1
    # 🔍 Score: 0.7008
    # 📘 Subject: None - student_list
    # 📄 Content:
    # Họ và tên: Đỗ Tuấn Minh. Mã số sinh viên: DE190415. Email: tuanminhdo1203@gmail.com.""")

    # --- Step 4: Hiển thị kết quả
    for i, hit in enumerate(hits, 1):  # bắt đầu từ 1
        print(f"\n⭐ Top {i}")
        print(f"🔍 Score: {hit.score:.4f}")
        print(f"📘 Subject: {hit.payload.get('subject_code')} - {hit.payload.get('type')}")
        print(f"📄 Content:\n{hit.payload.get('content')}")
        print("--------")


    # --- VÒNG LẶP HIỂN THỊ KẾT QUẢ (PHẦN KẾT HỢP ĐÃ SỬA LỖI RATE LIMIT) ---

    print("\n==============================================")
    print("✅ KẾT QUẢ TÌM KIẾM VÀ TÓM TẮT BẰNG GEMINI ✅")
    print("==============================================")

    if not hits:
        print("Không tìm thấy kết quả phù hợp.")
    else:
        retrieved_chunks=''
        for i, hit in enumerate(hits, 1):
            original_content = hit.payload.get('content', '')
            retrieved_chunks=retrieved_chunks + original_content
        
            

            # [THÊM MỚI] Tạm dừng 4 giây sau mỗi lần gọi API để tránh Rate Limit
            # Áp dụng cho đến khi chỉ còn kết quả cuối cùng
            # if i < len(hits):
            #     # print("... Tạm dừng 4 giây ...")
            #     time.sleep(4)
        print("\n[*] Gemini đang tóm tắt...")
            
        # Giữ nguyên cách gọi hàm với API key được gõ trực tiếp như code gốc của bạn
        summary = summarize_with_gemini(
            original_content, 
            gemini_api_key,
            retrieved_chunks=retrieved_chunks,
            user_query=query
        )
        
        print(f"✨ Tóm tắt của Gemini: {summary}")
        print("------------------------------------------")

In [ ]:
search(querry='môn học có điểm bonus cho bài báo')

In [ ]:
# from qdrant_client.http import models
# collection_name = "flm_fap"

# # Scroll toàn bộ điểm có payload (bạn có thể chia batch nếu lớn)
# scroll_result = client.scroll(
#     collection_name=collection_name,
#     with_payload=True,
#     limit=10000  # Tùy chỉnh nếu nhiều hơn
# )

# # Lọc các point có SubjectCode bắt đầu bằng "AI17_COM"
# points_to_delete = [
#     point.id
#     for point in scroll_result[0]
#     if isinstance(point.payload.get("subject_code"), str)
#     and point.payload["subject_code"].startswith("PHE_COM")
# ]
# print(points_to_delete)
# # Xoá các point
# if points_to_delete:
#     client.delete(
#         collection_name=collection_name,
#         points_selector=models.PointIdsList(points=points_to_delete)
#     )
#     print(f"Deleted {len(points_to_delete)} points.")
# else:
#     print("No points matched 'SubjectCode' starting with 'PHE_COM'.")

['325dcd8b-71ab-49a6-b831-59e8f47836b0', 'a72f02fb-ae0c-483c-b95b-6675ec2d04a6', 'da2f4d05-3b1d-4c87-9bb5-c51ce7c9bea2']
Deleted 3 points.


In [ ]:
# for point in points:
#     print(point.payload.keys())


## GEMINI giúp phân tích QUERY

In [ ]:
import re
def extract_json_from_markdown(text):
    match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        json_str = match.group(1)
        return json.loads(json_str)
    else:
        raise ValueError("Không tìm thấy JSON trong markdown block.")
    

def build_classification_prompt(query: str) -> str:
    return f"""
You are an AI assistant helping classify a student's academic query.

## Task:
Given a query (possibly in Vietnamese), you need to:
1. **Translate the query to English** first.
2. Determine the **type** of information being asked (from a fixed set of types).
3. Identify any clearly related **subject codes**. Only return subjects if you're confident.
4. Estimate the **semester** (0–9) if it is clearly implied. Omit this field if unsure.

## Types:
{json.dumps(TYPE_DESCRIPTIONS, indent=2)}

## Subjects:
You are provided a mapping of subject codes to their full names:
{json.dumps(subject_map, indent=2)}

## Output format:
Return a JSON object with these fields:
- `"type"`: One of the predefined type keys.
- `"subjects"`: A list of subject codes (e.g., ["SEG301", "SSL101c"]). Leave empty if not confident.
- `"semester"`: Integer from 0 to 9, **only if confident**. Omit this field if unsure.
- `"query_en"`: The English translation of the input query.

## Original Query:
"{query}"

## Output JSON:
""".strip()






def analyze_intent_with_gemini(api_key: str, model: str = "models/gemini-2.0-flash", query='') -> str:
    """
    Tóm tắt nội dung sử dụng Gemini API qua REST request.
    """
    url = f"https://generativelanguage.googleapis.com/v1beta/{model}:generateContent?key={api_key}"
    headers = {"Content-Type": "application/json"}
    
    prompt = build_classification_prompt(query)

    
    data = {"contents": [{"parts": [{"text": prompt}]}]}
    
    try:
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        result = response.json()
        summary = result["candidates"][0]["content"]["parts"][0]["text"]
        return extract_json_from_markdown(summary)
    except requests.exceptions.RequestException as e:
        print(f"Lỗi gọi API: {e}")
        return "Lỗi khi gọi API tóm tắt."
    except (KeyError, IndexError) as e:
        print(f"Lỗi khi xử lý response từ API: {e}. Response: {response.text}")
        return "Không thể tóm tắt nội dung do lỗi xử lý response."
    






In [ ]:
query = "AIL và DPL"

# query_en = query  # hoặc dùng translate nếu cần
# query_en = translate_vi_to_en_google(query)
analyze=analyze_intent_with_gemini(gemini_api_key,query=query) 
print(analyze)
query_en=analyze["query_en"]
detected_type=analyze["type"]
detected_subject=analyze["subjects"] 
detected_semester=analyze["semester"] if 'semester' in analyze.keys() else ''

if detected_type=='student_list':
    query_vec = embedder.embed([query])[0]
else: 
    query_vec = embedder.embed([query_en])[0]


query_filter = {"should": [], "must":[]}
if detected_type:
    print("Loại detect được là:")
    print(detected_type)
    query_filter["must"].append({"key": "type", "match": {"value": detected_type}})

if detected_subject:
    for subject_code in detected_subject:
        print(subject_code)
        query_filter["should"].append({
            "key": "subject_code",
            "match": {"value": subject_code}
        })

if detected_semester:
    print("Kì detect được là:")
    print(detected_semester)
    query_filter["should"].append({"key": "semester", "match": {"value": detected_semester}})

hits = client.search(
    collection_name="flm_fap",
    query_vector=query_vec.tolist(),
    limit=10,
    query_filter=query_filter if query_filter["must"] or query_filter["should"] else None
)
# print("""⭐ Top 1
# 🔍 Score: 0.7008
# 📘 Subject: None - student_list
# 📄 Content:
# Họ và tên: Đỗ Tuấn Minh. Mã số sinh viên: DE190415. Email: tuanminhdo1203@gmail.com.""")

# --- Step 4: Hiển thị kết quả
for i, hit in enumerate(hits, 1):  # bắt đầu từ 1
    print(f"\n⭐ Top {i}")
    print(f"🔍 Score: {hit.score:.4f}")
    print(f"📘 Subject: {hit.payload.get('subject_code')} - {hit.payload.get('type')}")
    print(f"📄 Content:\n{hit.payload.get('content')}")
    print("--------")

{'type': 'overview', 'subjects': ['AIL303m', 'DPL302m'], 'query_en': 'AIL and DPL'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loại detect được là:
overview
AIL303m
DPL302m


C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_15460\2395562874.py:37: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(



⭐ Top 1
🔍 Score: 0.5185
📘 Subject: DPL302m - overview
📄 Content:
TYPE: overview
Subject Code: DPL302m
Subject Name: Deep Learning_Học sâu
Degree Level: Bachelor | Credits: 6 | Semester: 5
Belong To Combo: None
Pre-requisites: AIL303m
Scoring Scale: 10.0 | Min Avg Mark to Pass: 5.0
Approved: True on 11/22/2024
Subject Link: https://flm.fpt.edu.vn/gui/role/student/Syllabuses.aspx?subCode=DPL302m&curriculumID=2347

--- TIME ALLOCATION ---
Study hour (300 h) = 90 h contact hours + 1 h final exam + 209 h self-study
--- TIME ALLOCATION END ---

--- DESCRIPTION ---
- In this course, student will build and train neural network architectures such as Convolutional Neural Networks, Recurrent Neural Networks, LSTMs, Transformers, and learn how to make them better with strategies such as Dropout, Batch Norm, Xavier/He initialization, and more. Get ready to master theoretical concepts and their industry applications using Python and Tensor Flow and tackle real-world cases such as speech recognition

### Tách nhỏ phần phân tích intent ra giảm phụ thuộc gemini

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import torch

# Load mô hình và tokenizer đã fine-tune
model = DistilBertForSequenceClassification.from_pretrained("D:\Learn\Semester_5\SEG301\DEMO_local\FLM\intent_model")
tokenizer = DistilBertTokenizerFast.from_pretrained("D:\Learn\Semester_5\SEG301\DEMO_local\FLM\intent_model")
model.eval()


<>:5: SyntaxWarning: invalid escape sequence '\L'
<>:6: SyntaxWarning: invalid escape sequence '\L'
<>:5: SyntaxWarning: invalid escape sequence '\L'
<>:6: SyntaxWarning: invalid escape sequence '\L'
C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_24540\83033018.py:5: SyntaxWarning: invalid escape sequence '\L'
  model = DistilBertForSequenceClassification.from_pretrained("D:\Learn\Semester_5\SEG301\DEMO_local\FLM\intent_model")
C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_24540\83033018.py:6: SyntaxWarning: invalid escape sequence '\L'
  tokenizer = DistilBertTokenizerFast.from_pretrained("D:\Learn\Semester_5\SEG301\DEMO_local\FLM\intent_model")


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
import torch
import requests
def predict_type(query_en: str) -> str:
    inputs = tokenizer(query_en, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=1).item()
    return model.config.id2label[pred]

def translate_with_gemini(text: str, api_key: str, model: str = "models/gemini-2.0-flash") -> str:
    url = f"https://generativelanguage.googleapis.com/v1beta/{model}:generateContent?key={api_key}"
    headers = {"Content-Type": "application/json"}
    prompt = f"just answer shortly, Translate this Vietnamese query to English:\n{text}"
    data = {"contents": [{"parts": [{"text": prompt}]}]}
    try:
        res = requests.post(url, headers=headers, json=data)
        res.raise_for_status()
        return res.json()['candidates'][0]['content']['parts'][0]['text'].strip()
    except Exception as e:
        print(f"Lỗi dịch Gemini: {e}")
        return "[Translation failed]"

# === 3. Test ===

def extract_semester(query: str) -> int | None:
    patterns = [
        r"k[ỳì]\s*(\d+)",      # kỳ 5
        r"semester\s*(\d+)",   # semester 3
        r"term\s*(\d+)",       # term 2
        r"k[ỳì]\s*cuối",       # kỳ cuối → có thể gán là 9
        r"k[ỳì]\s*đ[âầu]",     # kỳ đầu → 0
    ]
    
    for p in patterns:
        match = re.search(p, query.lower())
        if match:
            try:
                return int(match.group(1))
            except:
                if "cuối" in p: return 9
                if "đầu" in p: return 0
    return None

def analyze_intent(query_vi: str, api_key: str) -> dict:
    query_en = translate_with_gemini(query_vi, api_key)
    intent = predict_type(query_en)
    subjects = detect_subject(query_en)
    semester = extract_semester(query_vi)

    result = {
        "type": intent,
        "subjects": subjects,
        "query_en": query_en
    }
    if semester is not None:
        result["semester"] = semester

    return result

# analyze_intent('môn nào liên quan tới xử lí ảnh', subject_map=subject_map, api_key=api_key)

In [ ]:
query = "môn AIL là môn tiên quyết của môn"

# query_en = query  # hoặc dùng translate nếu cần
# query_en = translate_vi_to_en_google(query)
analyze=analyze_intent(query_vi=query, api_key=gemini_api_key) 
print(analyze)
query_en=analyze["query_en"]
detected_type=analyze["type"]
detected_subject=analyze["subjects"] 
detected_semester=analyze["semester"] if 'semester' in analyze.keys() else ''

if detected_type=='student_list':
    query_vec = embedder.embed([query])[0]
else: 
    query_vec = embedder.embed([query_en])[0]


query_filter = {"should": [], "must":[]}
if detected_type:
    print("Loại detect được là:")
    print(detected_type)
    query_filter["must"].append({"key": "type", "match": {"value": detected_type}})

if detected_subject:
    for subject_code in detected_subject:
        print(subject_code)
        query_filter["should"].append({
            "key": "subject_code",
            "match": {"value": subject_code[0]}
        })

if detected_semester:
    print("Kì detect được là:")
    print(detected_semester)
    query_filter["should"].append({"key": "semester", "match": {"value": detected_semester}})

hits = client.search(
    collection_name="flm_fap",
    query_vector=query_vec.tolist(),
    limit=10,
    query_filter=query_filter if query_filter["must"] or query_filter["should"] else None
)
# print("""⭐ Top 1
# 🔍 Score: 0.7008
# 📘 Subject: None - student_list
# 📄 Content:
# Họ và tên: Đỗ Tuấn Minh. Mã số sinh viên: DE190415. Email: tuanminhdo1203@gmail.com.""")

# --- Step 4: Hiển thị kết quả
for i, hit in enumerate(hits, 1):  # bắt đầu từ 1
    print(f"\n⭐ Top {i}")
    print(f"🔍 Score: {hit.score:.4f}")
    print(f"📘 Subject: {hit.payload.get('subject_code')} - {hit.payload.get('type')}")
    print(f"📄 Content:\n{hit.payload.get('content')}")
    print("--------")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'type': 'overview', 'subjects': [], 'query_en': 'AIL is a prerequisite for [the] ... course.'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loại detect được là:
overview


C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_24540\3440503786.py:37: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(



⭐ Top 1
🔍 Score: 0.5899
📘 Subject: PEN - overview
📄 Content:
TYPE: overview
Subject Code: PEN
Subject Name: Preparation English_Tiếng Anh chuẩn bị
Degree Level: N/A | Credits: 0 | Semester: 0
Belong To Combo: None
Pre-requisites: None
Scoring Scale: N/A | Min Avg Mark to Pass: N/A
Approved: N/A on N/A
Subject Link: https://flm.fpt.edu.vn/gui/role/student/Syllabuses.aspx?subCode=PEN&curriculumID=2347

--- TIME ALLOCATION ---
N/A
--- TIME ALLOCATION END ---

--- DESCRIPTION ---

--- END DESCRIPTION ---
--- STUDENT TASKS ---

--- END STUDENT TASKS ---
--- TOOLS ---

--- END TOOLS ---
--- NOTE ---

--- END NOTE ---

--------

⭐ Top 2
🔍 Score: 0.5838
📘 Subject: AIL303m - overview
📄 Content:
TYPE: overview
Subject Code: AIL303m
Subject Name: Machine Learning_Học máy
Degree Level: Bachelor | Credits: 3 | Semester: 4
Belong To Combo: None
Pre-requisites: MAS291, MAI391, PFP191
Scoring Scale: 10.0 | Min Avg Mark to Pass: 5.0
Approved: True on 4/3/2024
Subject Link: https://flm.fpt.edu.vn/gui/r

In [ ]:
import json
import time
from tqdm import tqdm

# Load tập truy vấn
with open(r"D:\Learn\Semester_5\SEG301\DEMO_local\data_eval.json", "r", encoding="utf-8") as f:
    query_list = json.load(f)

reciprocal_ranks = []
processing_times = []

for query_data in tqdm(query_list):
    query_text = query_data["query_text"]
    query_type = query_data["query_type"]
    label_id = query_data["label_id"]

    # Start timer
    start_time = time.time()

    # --- Intent analysis
    analyze = analyze_intent(query_vi=query_text, api_key=gemini_api_key)
    query_en = analyze["query_en"]
    detected_type = analyze["type"]
    detected_subject = analyze.get("subjects", [])
    detected_semester = analyze.get("semester", "")

    # --- Embedding
    if detected_type == 'student_list':
        query_vec = embedder.embed([query_text])[0]
    else:
        query_vec = embedder.embed([query_en])[0]

    # --- Filtering
    query_filter = {"should": [], "must": []}
    if detected_type:
        query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    for subject_code in detected_subject:
        query_filter["should"].append({"key": "subject_code", "match": {"value": subject_code[0]}})
    if detected_semester:
        query_filter["should"].append({"key": "semester", "match": {"value": detected_semester}})

    # --- Search
    hits = client.search(
        collection_name="flm_fap",
        query_vector=query_vec.tolist(),
        limit=10,
        query_filter=query_filter if query_filter["must"] or query_filter["should"] else None
    )

    # End timer
    elapsed_time = time.time() - start_time
    processing_times.append(elapsed_time)

    # --- Tính Reciprocal Rank
    rr = 0.0
    for rank, hit in enumerate(hits, 1):
        chunk_id = hit.id
        if chunk_id == label_id:
            rr = 1.0 / rank
            break
    reciprocal_ranks.append(rr)

# --- Tính MRR và thời gian trung bình
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
avg_time = sum(processing_times) / len(processing_times)

print(f"\n✅ MRR: {mrr:.4f}")
print(f"⏱️ Thời gian trung bình mỗi truy vấn: {avg_time:.4f} giây")


## Hàm kết hợp hoàn chỉnh

In [ ]:
# query = "các môn học có điểm bonus cho bài báo được accept"
# query_en = query  # hoặc dùng translate nếu cần
# query_en = translate_vi_to_en_google(query)
def search_full(query=''):
    query=query
    query_filter = {"should": [], "must": []}
    query_en = ''
    detected_type = ''
    detected_subject = []
    detected_semester = ''
    try:
        # Gọi Gemini API phân tích intent
        analyze = analyze_intent_with_gemini(gemini_api_key, query=query)
        print("🔍 Analyze thành công từ Gemini:")
        print(analyze)

        # Check các trường bắt buộc có trong kết quả
        if "query_en" in analyze and "type" in analyze and "subjects" in analyze:
            query_en = analyze["query_en"]
            detected_type = analyze["type"]
            detected_subject = analyze["subjects"]
            detected_semester = analyze.get("semester", "")
            if detected_type =='student_list' or detected_type =='guide': 
                detected_subject = []
            if detected_subject:
                for subject_code in detected_subject:
                    print(f"🎯 Mã môn phát hiện: {subject_code}")
                    query_filter["should"].append({
                        "key": "subject_code",
                        "match": {"value": subject_code}
                    })
        else:
            raise ValueError("Missing fields in Gemini response")

    except Exception as e:
        print(f"⚠️ Lỗi khi gọi hoặc xử lý Gemini: {e}")
        print("↩️ Fallback sang xử lý thủ công")

        query_en = translate_vi_to_en_local(query)
        print('📝 Query dịch: ', query_en)

        detected_type = detect_type_by_embedding(query_en)
        detected_subject = detect_subject(query_en)
        if detected_type =='student_list' or detected_type =='guide': 
            detected_subject = []
        if detected_subject:
            for subject_code in detected_subject:
                code = subject_code[0] if isinstance(subject_code, (list, tuple)) else subject_code
                print(f"🎯 Mã môn phát hiện (thủ công): {code}")
                query_filter["should"].append({
                    "key": "subject_code",
                    "match": {"value": code}
                })

    if detected_type=='student_list':
        query_vec = embedder.embed([query])[0]
    else: 
        query_vec = embedder.embed([query_en])[0]


    # query_filter = {"must": []}
    # if detected_type:
    #     query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    # if detected_subject:
    #     query_filter["must"].append({"key": "subject_code", "match": {"value": detected_subject}})
    
 
    if detected_type:
        print("Loại detect được là:")
        print(detected_type)
        query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    # if detected_subject:
    #     for subject_code in detected_subject:
    #         print(subject_code)
    #         query_filter["should"].append({
    #             "key": "subject_code",
    #             "match": {"value": subject_code}
    #         })
    if detected_semester:
        print("Kì detect được là:")
        print(detected_semester)
        query_filter["should"].append({"key": "semester", "match": {"value": detected_semester}})

        
    print(query_filter)
    hits = client.search(
        collection_name="FINAL",
        query_vector=query_vec.tolist(),
        limit=30,
        query_filter=query_filter if query_filter["must"] or query_filter["should"] else None
    )
    # print("""⭐ Top 1
    # 🔍 Score: 0.7008
    # 📘 Subject: None - student_list
    # 📄 Content:
    # Họ và tên: Đỗ Tuấn Minh. Mã số sinh viên: DE190415. Email: tuanminhdo1203@gmail.com.""")

    # --- Step 4: Hiển thị kết quả
    for i, hit in enumerate(hits, 1):  # bắt đầu từ 1
        print(f"\n⭐ Top {i}")
        print(f"🔍 Score: {hit.score:.4f}")
        print(f"📘 Subject: {hit.payload.get('subject_code')} - {hit.payload.get('type')}")
        print(f"📄 Content:\n{hit.payload.get('content')}")
        print("--------")
    retrieved_chunks=''
    
    for i, hit in enumerate(hits, 1):
        original_content = hit.payload.get('content', '')
        retrieved_chunks=retrieved_chunks + original_content

        

    
    print("\n[*] Gemini đang tóm tắt...")
        
    # Giữ nguyên cách gọi hàm với API key được gõ trực tiếp như code gốc của bạn
    summary = summarize_with_gemini(
        gemini_api_key,
        retrieved_chunks=retrieved_chunks,
        user_query=query
    )

    print(f"✨ Tóm tắt của Gemini: {summary}")
    print("------------------------------------------")

In [ ]:
search_full('Tỷ lệ PE của môn Nhập môn kĩ thuật phần mềm?')

🔍 Analyze thành công từ Gemini:
{'type': 'assessment', 'subjects': ['SWE201c'], 'query_en': 'What is the percentage weight of the Progress Exam (PE) for Introduction to Software Engineering?'}
🎯 Mã môn phát hiện: SWE201c


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loại detect được là:
assessment
{'should': [{'key': 'subject_code', 'match': {'value': 'SWE201c'}}], 'must': [{'key': 'type', 'match': {'value': 'assessment'}}]}


C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_33664\3369245524.py:86: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(



⭐ Top 1
🔍 Score: 0.7429
📘 Subject: SWE201c - assessment
📄 Content:
TYPE: assessment
Subject: SWE201c - Introduction to Software Engineering_Nhập môn kĩ thuật phần mềm
Category: PE | Part: 1 | Weight: 40.0%
Question Type: N/A
Knowledge and Skill: All studied courses.
Grading Guide: by Exam Board
Completion Criteria: 4
Duration: 120'
Note: Customized from the exercises of this specialization.
--------

⭐ Top 2
🔍 Score: 0.7165
📘 Subject: SWE201c - assessment
📄 Content:
TYPE: assessment
Subject: SWE201c - Introduction to Software Engineering_Nhập môn kĩ thuật phần mềm
Category: TE | Part: 1 | Weight: 60.0%
Question Type: Computer gradable
Knowledge and Skill: All studied courses. Each module of course contributes 2-3 questions.
Grading Guide: by Exam Board
Completion Criteria: 4
Duration: 60'
Note: Customized from the quizzes of this specialization.
--------

[*] Gemini đang tóm tắt...
✨ Tóm tắt của Gemini: Tỷ lệ PE của môn Nhập môn kĩ thuật phần mềm là 40%.
------------------------------

### Thử phân tích intent bằng model nhỏ

In [ ]:
# query = "các môn học có điểm bonus cho bài báo được accept"
# query_en = query  # hoặc dùng translate nếu cần
# query_en = translate_vi_to_en_google(query)
def search_full(query=''):
    query=query
    query_filter = {"should": [], "must": []}
    query_en = ''
    detected_type = ''
    detected_subject = []
    detected_semester = ''
    try:
        # Gọi Gemini API phân tích intent
        analyze = analyze_intent(query_vi=query, api_key=gemini_api_key) 
        print("🔍 Analyze thành công từ Model_Finetune:")
        print(analyze)

        # Check các trường bắt buộc có trong kết quả
        if "query_en" in analyze and "type" in analyze and "subjects" in analyze:
            query_en = analyze["query_en"]
            detected_type = analyze["type"]
            detected_subject = analyze["subjects"]
            detected_semester = analyze.get("semester", "")
            if detected_type =='student_list' or detected_type =='guide': 
                detected_subject = []
            if detected_subject:
                for subject_code in detected_subject:
                    print(f"🎯 Mã môn phát hiện: {subject_code}")
                    query_filter["should"].append({
                        "key": "subject_code",
                        "match": {"value": subject_code[0]}
                    })
        else:
            raise ValueError("Missing fields in Gemini response")

    except Exception as e:
        print(f"⚠️ Lỗi khi gọi hoặc xử lý Gemini: {e}")
        print("↩️ Fallback sang xử lý thủ công")

        query_en = translate_vi_to_en_local(query)
        print('📝 Query dịch: ', query_en)

        detected_type = detect_type_by_embedding(query_en)
        detected_subject = detect_subject(query_en)
        if detected_type =='student_list' or detected_type =='guide': 
            detected_subject = []
        if detected_subject:
            for subject_code in detected_subject:
                code = subject_code[0] if isinstance(subject_code, (list, tuple)) else subject_code
                print(f"🎯 Mã môn phát hiện (thủ công): {code}")
                query_filter["should"].append({
                    "key": "subject_code",
                    "match": {"value": code}
                })

    if detected_type=='student_list':
        query_vec = embedder.embed([query])[0]
    else: 
        query_vec = embedder.embed([query_en])[0]


    # query_filter = {"must": []}
    # if detected_type:
    #     query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    # if detected_subject:
    #     query_filter["must"].append({"key": "subject_code", "match": {"value": detected_subject}})
    
 
    if detected_type:
        print("Loại detect được là:")
        print(detected_type)
        query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
    # if detected_subject:
    #     for subject_code in detected_subject:
    #         print(subject_code)
    #         query_filter["should"].append({
    #             "key": "subject_code",
    #             "match": {"value": subject_code}
    #         })
    if detected_semester:
        print("Kì detect được là:")
        print(detected_semester)
        query_filter["should"].append({"key": "semester", "match": {"value": detected_semester}})

        
    print(query_filter)
    hits = client.search(
        collection_name="FINAL",
        query_vector=query_vec.tolist(),
        limit=30,
        query_filter=query_filter if query_filter["must"] or query_filter["should"] else None
    )
    # print("""⭐ Top 1
    # 🔍 Score: 0.7008
    # 📘 Subject: None - student_list
    # 📄 Content:
    # Họ và tên: Đỗ Tuấn Minh. Mã số sinh viên: DE190415. Email: tuanminhdo1203@gmail.com.""")

    # --- Step 4: Hiển thị kết quả
    for i, hit in enumerate(hits, 1):  # bắt đầu từ 1
        print(f"\n⭐ Top {i}")
        print(f"🔍 Score: {hit.score:.4f}")
        print(f"📘 Subject: {hit.payload.get('subject_code')} - {hit.payload.get('type')}")
        print(f"📄 Content:\n{hit.payload.get('content')}")
        print("--------")
    retrieved_chunks=''
    
    for i, hit in enumerate(hits, 1):
        original_content = hit.payload.get('content', '')
        retrieved_chunks=retrieved_chunks + original_content

        

    
    print("\n[*] Gemini đang tóm tắt...")
        
    # Giữ nguyên cách gọi hàm với API key được gõ trực tiếp như code gốc của bạn
    summary = summarize_with_gemini(
        gemini_api_key,
        retrieved_chunks=retrieved_chunks,
        user_query=query
    )

    print(f"✨ Tóm tắt của Gemini: {summary}")
    print("------------------------------------------")

In [ ]:
search_full('các môn kì 5')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔍 Analyze thành công từ Model_Finetune:
{'type': 'overview', 'subjects': [], 'query_en': 'Subjects in semester 5', 'semester': 5}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loại detect được là:
overview
Kì detect được là:
5
{'should': [{'key': 'semester', 'match': {'value': 5}}], 'must': [{'key': 'type', 'match': {'value': 'overview'}}]}


C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_24540\60736174.py:86: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(



⭐ Top 1
🔍 Score: 0.6082
📘 Subject: SEG301 - overview
📄 Content:
TYPE: overview
Subject Code: SEG301
Subject Name: Search Engines_Công cụ tìm kiếm
Degree Level: Bachelor | Credits: 3 | Semester: 5
Belong To Combo: AI17_COM2.1: Topic on AI Applied research_Chủ đề Định hướng nghiên cứu ứng dụng TTNT
Pre-requisites: None
Scoring Scale: 10.0 | Min Avg Mark to Pass: 5.0
Approved: True on 4/9/2025
Subject Link: https://flm.fpt.edu.vn/gui/role/student/SyllabusDetails.aspx?sylid=12633

--- TIME ALLOCATION ---
45 h (60 sessions) contact hours + 1 h final exam + 104 h self-study
--- TIME ALLOCATION END ---

--- DESCRIPTION ---
- The theory, creation, and use of text-based search engines are covered in this lecture-based course. The main content includes experimental assessment, representation of information demands and documents, several significant retrieval models, and statistical features of the text. Besides, the course covers typical components of commercial search engines, such as federate

In [ ]:
client.create_payload_index(
    collection_name="FINAL",
    field_name="semester",
    field_schema="integer"
)


UpdateResult(operation_id=140, status=<UpdateStatus.COMPLETED: 'completed'>)

# Thử với model mistral

In [ ]:
from langchain_community.llms import Ollama

llm = Ollama(model="mistral")



 The United States of America (USA), commonly known as the United States or America, is a federal republic consisting of 50 states, a federal district (Washington D.C.), five major self-governing territories, and various possessions. It's located in North America between Canada and Mexico, spanning an area of approximately 3.5 million square miles (9.1 million square kilometers). The capital is Washington D.C., although it isn't a state but the district where the federal government resides. The country was founded on July 4, 1776, declaring independence from Great Britain. It is a global leader in economics, science, technology, and culture.


In [ ]:
res = llm.invoke("""What is USA""")

print(res)

 The United States of America (USA), commonly known as the United States or America, is a country primarily located in North America. It is a federal republic consisting of 50 states, a federal district (Washington D.C.), five major self-governing territories, and various possessions.

The contiguous 48 states and Washington D.C. are in central North America between the Pacific and Atlantic Oceans, bordered by Canada to the north and Mexico to the south. The state of Alaska is in the far northwestern part of North America and is separated from the contiguous states by Canada. Hawaii is an archipelago in the mid-Pacific Ocean.

The United States was founded on July 4, 1776, after declaring independence from Great Britain. It's one of the world's most influential nations, playing a leading role in global politics, economics, science, technology, and culture. The capital is Washington, D.C., and its most populous city is New York City. English is the de facto national language, although t

In [ ]:
# # --- Mistral API Configuration ---
# API_KEY = 
# MODEL = "mistralai/mistral-7b-instruct:free"
# API_URL = "https://openrouter.ai/api/v1/chat/completions"

# def call_llama_api(messages, model=MODEL, api_key=API_KEY, max_tokens=512, temperature=0.2):
#     headers = {
#         "Content-Type": "application/json",
#         "Authorization": f"Bearer {api_key}",
#         "HTTP-Referer": "https://openrouter.ai/",
#         "X-Title": "FapChatbot"
#     }
#     data = {
#         "model": model,
#         "messages": messages,
#         "temperature": temperature,
#         "max_tokens": max_tokens
#     }
#     response = requests.post(API_URL, headers=headers, json=data)
#     if not response.ok:
#         print(f"Lỗi API: {response.status_code} - {response.text}")
#         return ""
#     return response.json()["choices"][0]["message"]["content"]

In [ ]:
def summarize_with_mistral_local(content: str, retrieved_chunks='', user_query='', llm=None) -> str:
    # Giới hạn context để tăng tốc
    max_chunks = 3000
    if len(retrieved_chunks) > max_chunks:
        retrieved_chunks = retrieved_chunks[:max_chunks] + "..."
    
    prompt = f"""
Answer the question based on the provided information:

Context: {retrieved_chunks}

Question: {user_query}

Provide a concise and accurate answer:
"""
    
    try:
        response = llm.invoke(prompt)
        return response.strip()
    except Exception as e:
        return "Error during summarization."
    

def analyze_intent_with_mistral_local(query='', llm=None):
    prompt = f"""
Classify this academic query: "{query}"

Question types:
{json.dumps(TYPE_DESCRIPTIONS, indent=2)}

Available subjects:
{json.dumps(subject_map, indent=2)}

Return a JSON object with these fields:
{{
  "type": "question_type",
  "subjects": ["subject_code"],
  "semester": semester_number,
  "query_en": "english_translation"
}}

Rules:
- "type": must be one of the predefined types above
- "subjects": list of subject codes, leave empty if not confident
- "semester": integer 0-9, only if clearly implied, omit if unsure
- "query_en": translate the original query to English

Your JSON response:
"""
    
    try:
        # Gọi trực tiếp, không cần messages format
        response = llm.invoke(prompt)
        print(f"🔍 Raw response: {response}")
        
        # Parse JSON nhanh
        try:
            return extract_json_from_markdown(response)
        except:
            import re
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                return json.loads(json_match.group(0))
            return None
                
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
def search_full_local(query='', llm=None):
    query_filter = {"should": [], "must": []}
    query_en = ''
    detected_type = ''
    detected_subject = []
    detected_semester = ''
    
    try:
        # 1. Phân tích query - tối ưu cho local
        analyze = analyze_intent_with_mistral_local(query=query, llm=llm)
        print("�� Analyze thành công từ Mistral Local:")
        print(analyze)

        if analyze and "query_en" in analyze and "type" in analyze and "subjects" in analyze:
            query_en = analyze["query_en"]
            detected_type = analyze["type"]
            detected_subject = analyze["subjects"]
            detected_semester = analyze.get("semester", "")
            
            if detected_type == 'student_list' or detected_type == 'guide': 
                detected_subject = []
                
            if detected_subject:
                for subject_code in detected_subject:
                    print(f"🎯 Mã môn phát hiện: {subject_code}")
                    query_filter["should"].append({
                        "key": "subject_code",
                        "match": {"value": subject_code}
                    })
        else:
            raise ValueError("Missing fields in response")

    except Exception as e:
        print(f"⚠️ Lỗi Mistral Local: {e}")
        print("↩️ Fallback sang xử lý thủ công")

        query_en = translate_vi_to_en_local(query)
        print('📝 Query dịch: ', query_en)

        detected_type = detect_type_by_embedding(query_en)
        detected_subject = detect_subject(query_en)
        if detected_type == 'student_list' or detected_type == 'guide': 
            detected_subject = []
        if detected_subject:
            for subject_code in detected_subject:
                code = subject_code[0] if isinstance(subject_code, (list, tuple)) else subject_code
                print(f"�� Mã môn phát hiện (thủ công): {code}")
                query_filter["should"].append({
                    "key": "subject_code",
                    "match": {"value": code}
                })

    if detected_type == 'student_list':
        query_vec = embedder.embed([query])[0]
    else: 
        query_vec = embedder.embed([query_en])[0]

    if detected_type:
        print("Loại detect được là:", detected_type)
        query_filter["must"].append({"key": "type", "match": {"value": detected_type}})
        
    if detected_semester:
        print("Kì detect được là:", detected_semester)
        query_filter["should"].append({"key": "semester", "match": {"value": detected_semester}})

    print("Query filter:", query_filter)
    
    # Giảm limit để tăng tốc
    hits = client.search(
        collection_name="FINAL",
        query_vector=query_vec.tolist(),
        limit=15,  # Giảm từ 30 xuống 15
        query_filter=query_filter if query_filter["must"] or query_filter["should"] else None
    )

    # Hiển thị kết quả
    for i, hit in enumerate(hits, 1):
        print(f"\n⭐ Top {i}")
        print(f"🔍 Score: {hit.score:.4f}")
        print(f"📘 Subject: {hit.payload.get('subject_code')} - {hit.payload.get('type')}")
        print(f"📄 Content:\n{hit.payload.get('content')}")
        print("--------")
    
    # Tóm tắt - chỉ lấy top 10 kết quả
    retrieved_chunks = ''
    for hit in hits[:10]:
        retrieved_chunks += hit.payload.get('content', '') + '\n'

    print("\n[*] Mistral Local đang tóm tắt...")
    
    summary = summarize_with_mistral_local(
        retrieved_chunks,
        retrieved_chunks=retrieved_chunks,
        user_query=query,
        llm=llm
    )

    print(f"✨ Tóm tắt của Mistral Local: {summary}")
    print("------------------------------------------")
    
    return {
        'query_translated': query_en,
        'detected_type': detected_type,
        'detected_subject': detected_subject,
        'detected_semester': detected_semester,
        'results': [hit.payload for hit in hits],
        'summary': summary
    }

In [ ]:
# # Khởi tạo LLM một lần
from langchain_community.llms import Ollama

llm = Ollama(
    model="mistral",
    temperature=0.1,
    num_ctx=4096,
    num_thread=4
)



In [ ]:
# Test
query = "các môn học có điểm bonus cho bài báo được accept"
result = search_full_local(query, llm)

🔍 Raw response:  {
  "type": "information_request",
  "subjects": ["CPV301"],
  "semester": 3,
  "query_en": "What is Computer Vision (CPV301) about in the third semester?"
}
�� Analyze thành công từ Mistral Local:
{'type': 'information_request', 'subjects': ['CPV301'], 'semester': 3, 'query_en': 'What is Computer Vision (CPV301) about in the third semester?'}
🎯 Mã môn phát hiện: CPV301


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loại detect được là: information_request
Kì detect được là: 3
Query filter: {'should': [{'key': 'subject_code', 'match': {'value': 'CPV301'}}, {'key': 'semester', 'match': {'value': 3}}], 'must': [{'key': 'type', 'match': {'value': 'information_request'}}]}


C:\Users\DO TUAN MINH\AppData\Local\Temp\ipykernel_6772\1338565732.py:69: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(



[*] Mistral Local đang tóm tắt...
✨ Tóm tắt của Mistral Local: Trong thực tế, việc có điểm bonus hoặc không cho bài báo được chấp nhận phụ thuộc vào các quy định của mỗi học liệu. Ví dụ, trong học liệu khoa học máy tính, bài báo có thể nhận điểm bonus nếu được chấp nhận vào các cuộc họp hoặc sự kiện chuyên khoa. Tuy nhiên, trong bài viết này, thông tin về các môn học có điểm bonus cho bài báo được accept không được cung cấp.
------------------------------------------


In [ ]:
import requests

resp = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "mistral",
        "prompt": "Giải thích hệ điều hành là gì?",
        "stream": True
    },
    stream=True
)

for line in resp.iter_lines():
    if line:
        print(line.decode().removeprefix("data: "), end="", flush=True)


{"model":"mistral","created_at":"2025-07-04T17:10:47.6831565Z","response":" H","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:47.8176581Z","response":"ệ","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:47.9501117Z","response":" ","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:48.0811893Z","response":"đ","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:48.2001774Z","response":"i","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:48.3242759Z","response":"ề","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:48.4363643Z","response":"u","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:48.540768Z","response":" h","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:48.649833Z","response":"à","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:48.7599436Z","response":"nh","done":false}{"model":"mistral","created_at":"2025-07-04T17:10:48.8781693Z","response":" (","done":fal

# DEMO với hugging face

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_id = "csebuetnlp/mT5_multilingual_XLSum"
  # hoặc đổi sang model khác

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)




c:\Users\DO TUAN MINH\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

c:\Users\DO TUAN MINH\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DO TUAN MINH\.cache\huggingface\hub\models--csebuetnlp--mT5_multilingual_XLSum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
c:\Users\DO TUAN MINH\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Tóm tắt: Hệ điều hành tài nguyên ở Việt Nam đang được công bố vào tháng này.


In [ ]:
text = """RA QUÂN Ở PHƯỜNG HÒA XUÂN MỚI: SIẾT CHẶT ANTT, KIỂM SOÁT ĐỐI TƯỢNG NGUY CƠ! 🔍🚨
🕔 5h sáng ngày 04/7/2025, Công an phường Hòa Xuân mới (sau sáp nhập từ 3 địa bàn: Hòa Xuân cũ, Hòa Phước, Hòa Châu) đồng loạt ra quân kiểm danh, kiểm diện, test nhanh ma túy đối với các đối tượng hình sự, ma túy có nguy cơ gây mất an ninh trật tự trên địa bàn.
👮‍♂️ 45 CBCS cùng 48 lực lượng An ninh tham gia bảo vệ ANTT ở cơ sở được huy động.
📋 Đã kiểm tra, test nhanh 22 đối tượng.
✅ 20 đối tượng âm tính.
❌ 2 đối tượng dương tính với ma túy – Công an phường đã lập hồ sơ xử lý theo quy định.
🎯 Đây là đợt ra quân quyết liệt đầu tiên sau sáp nhập, thể hiện rõ quyết tâm của lực lượng Công an phường Hòa Xuân mới trong việc giữ vững ANTT, làm sạch địa bàn ngay từ ban đầu."
"""
inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
summary_ids = model.generate(**inputs, max_new_tokens=100)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Tóm tắt:", summary)

Tóm tắt: Công an phường Hòa Xuân mới đồng loạt ra quân quyết liệt đầu tiên sau sáp nhập.
